# Recommend Items Frequently Purchased Together
This notebook demonstrates how recommending items that are frequently purchased together is effective. The current best scoring public notebook [here][1] recommends to customers those customers' last purchases and scores public LB 0.020. In this notebook here, we will begin with that idea and add recommending items that are frequently purchased together with a customers' previous purchaes. This notebook improves the LB and scores LB 0.021. This notebook's strategy is as follows:
* recommend items previously purchased [idea here][1]
* recommend items that are bought together with previous purchases [idea here][2]
* recommend popular items [idea here][1]

[1]: https://www.kaggle.com/hengzheng/time-is-our-best-friend-v2
[2]: https://www.kaggle.com/cdeotte/customers-who-bought-this-frequently-buy-this

# Environment

In [ ]:
from pathlib import Path
import pandas as pd

data_path = next(base / 'data' for base in (Path.cwd(), *Path.cwd().parents) if (base / 'data' / 'transactions_train.csv').is_file())

# Load Transactions, Reduce Memory
Discussion about reducing memory is [here][1]

[1]: https://www.kaggle.com/c/h-and-m-personalized-fashion-recommendations/discussion/308635

In [ ]:
import pandas as pd

In [ ]:
def compress_customer_id(series):
    return series.str[-16:].apply(lambda x: int(x, 16)).astype('uint64')

In [ ]:
train = pd.read_csv(data_path / 'transactions_train.csv')
train['customer_id'] = compress_customer_id(train['customer_id'])
train['article_id'] = train.article_id.astype('int32')
train.t_dat = pd.to_datetime(train.t_dat)
train = train[['t_dat','customer_id','article_id']]
train.to_parquet('train.pqt',index=False)
print( train.shape )
train.head()

# Find Each Customer's Last Week of Purchases
Our final predictions will have the row order from of our dataframe. Each row of our dataframe will be a prediction. We will create the `predictionstring` later by `train.groupby('customer_id').article_id.sum()`. Since `article_id` is a string, when we groupby sum, it will concatenate all the customer predictions into a single string. It will also create the string in the order of the dataframe. So as we proceed in this notebook, we will order the dataframe how we want our predictions ordered.

In [ ]:
tmp = train.groupby('customer_id').t_dat.max().reset_index()
tmp.columns = ['customer_id','max_dat']
train = train.merge(tmp,on=['customer_id'],how='left')
train['diff_dat'] = (train.max_dat - train.t_dat).dt.days
train = train.loc[train['diff_dat']<=6]
print('Train shape:',train.shape)

# (1) Recommend Most Often Previously Purchased Items
Rows are sorted by `ct` and then `t_dat` in descending order, so predictions contain more frequent purchases first and more recent purchases second.

In [ ]:
tmp = train.groupby(['customer_id','article_id'])['t_dat'].agg('count').reset_index()
tmp.columns = ['customer_id','article_id','ct']
train = train.merge(tmp,on=['customer_id','article_id'],how='left')
train = train.sort_values(['ct','t_dat'],ascending=False)
train = train.drop_duplicates(['customer_id','article_id'])
train = train.sort_values(['ct','t_dat'],ascending=False)
train.head()

# (2) Recommend Items Purchased Together
In my notebook [here][1], we compute a dictionary of items frequently purchased together. We will load and use that dictionary below. Note that we use the command `drop_duplicates` so that we don't recommend an item that the user has already bought and we have already recommended above.

We concatenate these rows after the rows containing customers' previous purchases. Therefore we will recommend previous items first and then items purchased together second. Note the trick to convert a column of int32 into a prediction string (using groupby agg str sum) is from notebook [here][2]

[1]: https://www.kaggle.com/cdeotte/customers-who-bought-this-frequently-buy-this
[2]: https://www.kaggle.com/hiroshisakiyama/recommending-items-recently-bought

In [ ]:
import numpy as np

In [ ]:
# # 2. 提取最近60天的数据 避免全量自连接造成OOM
# recent_date = train['t_dat'].max() - pd.Timedelta(days=60)
# df_recent = train[train['t_dat'] >= recent_date][['customer_id', 'article_id', 't_dat']].drop_duplicates()

# # 3. 自连接：找出同一天、同一个顾客购买的商品对 (A, B)
# df_pairs = pd.merge(
#     df_recent, 
#     df_recent, 
#     on=['customer_id', 't_dat'], 
#     suffixes=('', '_pair')
# )

# # 排除自己和自己配对的情况 (A != B)
# df_pairs = df_pairs[df_pairs['article_id'] != df_pairs['article_id_pair']]

# # 4. 统计 (A, B) 的共现频次
# pair_counts = (
#     df_pairs.groupby(['article_id', 'article_id_pair'])
#     .size()
#     .reset_index(name='count')
# )

# # 5. 对每个商品 A，按共现次数降序排序，取共现频次最高的那个商品 B
# pair_counts = pair_counts.sort_values(['article_id', 'count'], ascending=[True, False])
# best_pairs = pair_counts.drop_duplicates('article_id')

# # 6. 生成关联商品映射字典 pairs: {商品A: 商品B}
# pairs = dict(zip(best_pairs['article_id'], best_pairs['article_id_pair']))

# print(f"共挖掘出 {len(pairs)} 个商品的关联搭配。")

# # 7. 映射出伴侣商品列 article_id2（若某个商品没有共现商品，保留为 NaN）
# train['article_id2'] = train['article_id'].map(pairs)

# # 内存清理
# del df_recent, df_pairs, pair_counts, best_pairs

In [ ]:
from collections import defaultdict
from tqdm import tqdm

print("正在基于全量数据挖掘商品共现对 (pairs)...")

# 1. 仅提取计算需要的列，确保 article_id 为 目标数据类型
df = train[['customer_id', 'article_id', 't_dat']].drop_duplicates()
df['article_id'] = df['article_id'].astype('int32')

# 2. 按 (customer_id, t_dat) 分组，找出单次购物篮中的所有商品
# 过滤掉只买了 1 件商品的记录
basket = df.groupby(['customer_id', 't_dat'])['article_id'].apply(list)
basket = basket[basket.str.len() > 1]

# 3. 统计全量共现频次 (A, B) -> count
pair_counts = defaultdict(lambda: defaultdict(int))
for items in tqdm(basket, desc="Counting pairs"):
    n = len(items)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = items[i], items[j]
            pair_counts[a][b] += 1
            pair_counts[b][a] += 1

# 4. 对每个商品 A，找出共现频次最高的商品 B
pairs = {}
for a, related in pair_counts.items():
    if related:
        best_b = max(related.items(), key=lambda x: x[1])[0]
        pairs[a] = best_b

print(f"共覆盖 {len(pairs)} 个商品的伴侣推荐。")

# 5. 映射回训练集
train['article_id2'] = train['article_id'].map(pairs)

# 清理内存
del basket, pair_counts, df

In [ ]:
# RECOMMENDATION OF PAIRED ITEMS
train2 = train[['customer_id','article_id2']].copy()
train2 = train2.loc[train2.article_id2.notnull()]
train2 = train2.drop_duplicates(['customer_id','article_id2'])
train2 = train2.rename({'article_id2':'article_id'},axis=1)

In [ ]:
# CONCATENATE PAIRED ITEM RECOMMENDATION AFTER PREVIOUS PURCHASED RECOMMENDATIONS
train = train[['customer_id','article_id']]
train = pd.concat([train,train2],axis=0,ignore_index=True)
train.article_id = train.article_id.astype('int32')
train = train.drop_duplicates(['customer_id','article_id'])

In [ ]:
# CONVERT RECOMMENDATIONS INTO SINGLE STRING
train.article_id = ' 0' + train.article_id.astype('str')
preds = train.groupby('customer_id').article_id.sum().reset_index() 
preds.columns = ['customer_id','prediction']
preds.head()

2026.10.20下午

# (3) Recommend Last Week's Most Popular Items
After recommending previous purchases and items purchased together we will then recommend the 12 most popular items. Therefore if our previous recommendations did not fill up a customer's 12 recommendations, then it will be filled by popular items.

In [ ]:
train = pd.read_parquet('train.pqt')
train.t_dat = pd.to_datetime(train.t_dat)
train = train.loc[train.t_dat >= pd.to_datetime('2020-09-16')]
top12 = ' 0' + ' 0'.join(train.article_id.value_counts().index.astype('str')[:12])
print("Last week's top 12 popular items:")
print( top12 )

# Write Submission CSV
We will merge our predictions onto `sample_submission.csv` and submit to Kaggle.

In [ ]:
sub = pd.read_csv(data_path / 'sample_submission.csv')
sub = sub[['customer_id']]

# 替换 hex_to_int 为 apply(lambda x: int(x, 16))
sub['customer_id_2'] = sub['customer_id'].str[-16:].apply(lambda x: int(x, 16)).astype('uint64')

sub = sub.merge(
    preds.rename({'customer_id': 'customer_id_2'}, axis=1),
    on='customer_id_2', 
    how='left'
).fillna('')

del sub['customer_id_2']